## 안다르 공식몰 리뷰 데이터 전처리

In [94]:
# 데이터 처리 및 분석
import pandas as pd
import ast
import numpy as np
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import platform

# ── 한글 폰트 설정 ──
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# ── 출력 설정 ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


### 전처리 시작

In [95]:
# andar 리뷰 데이터 로드
df = pd.read_csv('./check_data/andar_homepage_review_final_s2024.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 638998 entries, 0 to 638997
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   collect_date       638998 non-null  str    
 1   platform           638998 non-null  str    
 2   review_id          638998 non-null  int64  
 3   product_id         638998 non-null  int64  
 4   product_name       638998 non-null  str    
 5   review_date        638998 non-null  str    
 6   year               638998 non-null  int64  
 7   month              638998 non-null  int64  
 8   content            638997 non-null  str    
 9   rating             638998 non-null  int64  
 10  helpful_count      638998 non-null  int64  
 11  has_image          638998 non-null  int64  
 12  purchase_option    551882 non-null  str    
 13  user_size          0 non-null       float64
 14  user_height        0 non-null       float64
 15  user_weight        0 non-null       float64
 16  user_height_g

In [96]:
df.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,user_size,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-21,자사몰,48724074,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-14,2026,4,"러닝 후 몸 풀어주는 용으로 아주 좋아요. 종아리나 등, 목 뒤에 사용하기 좋아요....",5,0,1,"[EJFMB-02DPK] 샤이 핑크, F (Free)",NaN,NaN,NaN,155~159cm,45~49kg,https://andar.co.kr/product/detail.html?produc...
1,2026-04-21,자사몰,48590531,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-07,2026,4,딱딱하네요\n저는 말랑말랑 한줄 알았어요(2026-04-07 20:15:59 에 등...,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,https://andar.co.kr/product/detail.html?produc...
2,2026-04-21,자사몰,48535840,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-05,2026,4,요가링이랑 폼롤러랑 같이 사서 집에서 잘 굴려주고 있어요 색깔도 너무 예쁘고 귀여워...,5,0,1,"[EJFMB-02DPK] 샤이 핑크, F (Free)",NaN,NaN,NaN,165~169cm,55~59kg,https://andar.co.kr/product/detail.html?produc...
3,2026-04-21,자사몰,48535469,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-05,2026,4,묵직하니 사용하기 편하네요 릴렉스 마사지 듀얼볼 Ⅱ 후기 정사이즈,5,0,0,"[EJFMB-02DPK] 샤이 핑크, F (Free)",NaN,NaN,NaN,160~164cm,50~54kg,https://andar.co.kr/product/detail.html?produc...
4,2026-04-21,자사몰,48521693,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-03,2026,4,시원하게 마사지가되요 여기저기 쉽게 문지르고 아픈 부위 꾹꾹 눌러주니 시원하네요 릴...,5,0,1,"[EJFMB-02DPK] 샤이 핑크, F (Free)",NaN,NaN,NaN,160~164cm,55~59kg,https://andar.co.kr/product/detail.html?produc...


In [97]:
# user_size 컬럼 삭제
df.drop(columns=['user_size'], inplace=True)

In [98]:
import re

opt = df['purchase_option']

# 컬럼 초기화
df['purchase_option_color'] = pd.NA
df['purchase_option_size']  = pd.NA

# ─────────────────────────────────────────────
# Step 1 : 레이블 기반  (색상= / 사이즈= 명시된 행)
#          세트 상품은 색상이 여러 개 → findall로 전부 추출 후 , 연결
# ─────────────────────────────────────────────
mask1 = opt.str.contains(r'색상\s*\d*\)?\s*=', na=False, regex=True)

df.loc[mask1, 'purchase_option_color'] = (
    opt[mask1]
    .str.findall(r'색상\s*\d*\)?\s*=\s*(?:\[.*?\])?\s*([^,]+)')
    .apply(lambda lst: ','.join(m.strip() for m in lst) if lst else pd.NA)
)
df.loc[mask1, 'purchase_option_size'] = (
    opt[mask1]
    .str.findall(r'사이즈\s*\d*\)?\s*=\s*([^,]+)')
    .apply(lambda lst: ','.join(m.strip() for m in lst) if lst else pd.NA)
)
print(f"Step 1 (레이블 기반)          : {mask1.sum():,}건 처리")

# ─────────────────────────────────────────────
# Step 2a : [SKU] 컬러, 사이즈  (purchase_option이 [로 시작)
# ─────────────────────────────────────────────
mask_rem = df['purchase_option_color'].isna() & opt.notna()
mask2a   = mask_rem & opt.str.match(r'^\[', na=False)

cleaned_2a = opt[mask2a].str.replace(r'\[.*?\]\s*', '', regex=True).str.strip()
df.loc[mask2a, 'purchase_option_color'] = cleaned_2a.str.split(',', n=1).str[0].str.strip()
df.loc[mask2a, 'purchase_option_size']  = cleaned_2a.str.split(',', n=1).str[1].str.strip()
print(f"Step 2a ([SKU] 컬러, 사이즈)   : {mask2a.sum():,}건 처리")

# ─────────────────────────────────────────────
# Step 2b : 사이즈, [SKU] 컬러  (중간에 , [ 패턴)
# ─────────────────────────────────────────────
mask_rem = df['purchase_option_color'].isna() & opt.notna()
mask2b   = mask_rem & opt.str.contains(r',\s*\[', na=False, regex=True)

df.loc[mask2b, 'purchase_option_size'] = (
    opt[mask2b].str.extract(r'^(.*?),\s*\[')[0].str.strip()
)
df.loc[mask2b, 'purchase_option_color'] = (
    opt[mask2b].str.replace(r'^.*?,\s*\[.*?\]\s*', '', regex=True).str.strip()
)
print(f"Step 2b (사이즈, [SKU] 컬러)   : {mask2b.sum():,}건 처리")

# ─────────────────────────────────────────────
# Step 3 : 앞 [ 누락된 SKU 패턴  →  사이즈, SKU] 컬러
# ─────────────────────────────────────────────
mask_rem = df['purchase_option_color'].isna() & opt.notna()
mask3    = mask_rem & opt.str.contains(r',\s*[A-Z0-9\-]+\]', na=False, regex=True)

df.loc[mask3, 'purchase_option_size'] = (
    opt[mask3].str.extract(r'^(.*?),\s*[A-Z0-9\-]+\]')[0].str.strip()
)
df.loc[mask3, 'purchase_option_color'] = (
    opt[mask3].str.replace(r'^.*?,\s*[A-Z0-9\-]+\]\s*', '', regex=True).str.strip()
)
print(f"Step 3 (앞 [ 누락 SKU)        : {mask3.sum():,}건 처리")

# ─────────────────────────────────────────────
# 미처리 최종 확인
# ─────────────────────────────────────────────
mask_rem = df['purchase_option_color'].isna() & opt.notna()
print(f"\n최종 미처리: {mask_rem.sum():,}건")
if mask_rem.sum() > 0:
    print(opt[mask_rem].value_counts().head(10))

Step 1 (레이블 기반)          : 184,129건 처리
Step 2a ([SKU] 컬러, 사이즈)   : 50,660건 처리
Step 2b (사이즈, [SKU] 컬러)   : 316,584건 처리
Step 3 (앞 [ 누락 SKU)        : 509건 처리

최종 미처리: 0건


In [99]:
# ─────────────────────────────────────────────
# 후처리: 컬러 정제 (토큰별 개별 정제, 중복 유지)
# ─────────────────────────────────────────────

fix_words  = r'숏|롱|미니|스탠다드핏|스탠다드|9부|8\.2부|4\.5부|3\.5부|퍼포먼스|후크형|노후크형|슬림핏|오버핏|볼륨업|HOLE|SOLID|맨즈'
_prefix = re.compile(rf'^({fix_words})\s*')
_suffix = re.compile(rf'\s*({fix_words})$')
_sku1   = re.compile(r'\[.*?\]')
_sku2   = re.compile(r'^[A-Z0-9\-]+\]\s*')
_paren  = re.compile(r'\(.*?\)')
_space  = re.compile(r'\s+')

def _clean_token(c):
    c = _sku1.sub('', c)
    c = _sku2.sub('', c)
    c = _paren.sub('', c)
    c = _prefix.sub('', c)
    c = _suffix.sub('', c)
    return _space.sub('', c).strip()

def clean_multicolor(s):
    if pd.isna(s) or s == '':
        return pd.NA
    tokens = [_clean_token(t) for t in s.replace('&', ',').split(',')]
    tokens = [t for t in tokens if t]   # 빈 문자열만 제거, 중복은 유지
    return ','.join(tokens) if tokens else pd.NA

df['purchase_option_color'] = df['purchase_option_color'].apply(clean_multicolor)

print(f"purchase_option_color non-null: {df['purchase_option_color'].notna().sum():,} ({df['purchase_option_color'].notna().mean():.1%})")
print(f"고유 컬러 수: {df['purchase_option_color'].nunique():,}개")
print()
print(df['purchase_option_color'].value_counts().head(20))

purchase_option_color non-null: 551,882 (86.4%)
고유 컬러 수: 27,644개

purchase_option_color
블랙          110057
블랙,블랙        15545
화이트          14311
블루            4681
네이비           4140
스톤블루          3603
얼스베이지         3556
화이트머스크        3076
프레쉬크림         2779
모카베이지         2588
그레이           2465
멜란지그레이        2409
믹스            2330
스톤카키          2291
애쉬블루          2285
오보시디안그레이      2200
실버베이지         2197
파스텔민트         2104
다크애쉬          2093
딥차콜           2059
Name: count, dtype: int64


In [100]:
col = df['purchase_option_color'].dropna()

# ① 숫자 포함 → 사이즈 정보 혼입 의심
mask_num = col.str.contains(r'\d', regex=True)
print(f"[숫자 포함] {mask_num.sum():,}건")
print(col[mask_num].value_counts().head(10))

# ② 대문자 영문 포함 → SKU 잔재 의심
mask_upper = col.str.contains(r'[A-Z]', regex=True)
print(f"\n[대문자 영문 포함] {mask_upper.sum():,}건")
print(col[mask_upper].value_counts().head(10))

# ③ 특수문자 잔재 ([, ], =, ,)
mask_special = col.str.contains(r'[\[\]=,]', regex=True)
print(f"\n[특수문자 잔재] {mask_special.sum():,}건")
print(col[mask_special].value_counts().head(10))

# ④ 비정상적으로 긴 컬러명 (10자 초과)
mask_long = col.str.len() > 10
print(f"\n[10자 초과 컬러명] {mask_long.sum():,}건")
print(col[mask_long].value_counts().head(10))

[숫자 포함] 83건
purchase_option_color
다이브블루,4.5부다크애쉬블루    13
다이브블루,3.5부모아나블루      8
딥라군,4.5부고블린블루        8
다이브블루,4.5부고블린블루      8
다이브블루,4.5부그로토네이비     6
딥라군,4.5부앤틱카키         5
블랙,4.5부블랙            4
블랙,3.5부블랙            3
딥라군,4.5부오이스터차콜       3
3.5부모아나블루,블랙         2
Name: count, dtype: int64

[대문자 영문 포함] 0건
Series([], Name: count, dtype: int64)

[특수문자 잔재] 186,029건
purchase_option_color
블랙,블랙           15545
블랙,앤트러사이트그레이     1625
블랙,블랙,블랙         1611
블랙,화이트           1553
블랙,마운틴뷰          1466
화이트,화이트          1370
화이트,블랙           1234
블랙,화이트머스크        1229
화이트머스크,블랙         923
블랙,웜베이지           923
Name: count, dtype: int64

[10자 초과 컬러명] 98,415건
purchase_option_color
블랙,앤트러사이트그레이              1625
블랙,오보시디안그레이                833
블랙,마운틴뷰,오트밀베이지             769
앤트러사이트그레이,블랙               680
화이트,화이트,화이트                626
마쉬멜로우,마쉬멜로우                390
스킨베이지,스킨베이지                385
클린화이트,클린화이트,클린화이트          356
스모키우드브라운,허쉬드그레이,바질머스타드     318
스모키우드브라운,블랙                309
Name: count, d

In [101]:
# 잔여 숫자+부 패턴 추가 정제 (4.5부, 3.5부 등)
_num_boo = re.compile(r'\d+\.?\d*부\s*')

mask_num = df['purchase_option_color'].str.contains(r'\d', na=False, regex=True)
print(f"처리 전: {mask_num.sum():,}건")

df.loc[mask_num, 'purchase_option_color'] = (
    df.loc[mask_num, 'purchase_option_color'].apply(
        lambda s: ','.join(
            t for t in [_num_boo.sub('', tok).strip() for tok in s.split(',')]
            if t
        ) if pd.notna(s) else pd.NA
    )
)

mask_num2 = df['purchase_option_color'].str.contains(r'\d', na=False, regex=True)
print(f"처리 후: {mask_num2.sum():,}건")
if mask_num2.sum() > 0:
    print(df.loc[mask_num2, 'purchase_option_color'].value_counts().head(10))

처리 전: 83건
처리 후: 0건


In [102]:
# 정렬 통일 시 고유 컬러 수 미리 확인 (실제 변경 없음)
sorted_preview = df['purchase_option_color'].dropna().apply(
    lambda s: ','.join(sorted(s.split(','))) if ',' in s else s
)

current  = df['purchase_option_color'].nunique()
after    = sorted_preview.nunique()
single   = df['purchase_option_color'].dropna()
multi_mask = single.str.contains(',', regex=False)

print(f"현재 고유 컬러 수     : {current:,}개")
print(f"정렬 후 예상 고유 수  : {after:,}개  (감소 {current - after:,}개)")
print()
print(f"단품 컬러 행 수       : {(~multi_mask).sum():,}건")
print(f"멀티컬러 행 수        : {multi_mask.sum():,}건")
print(f"  멀티 고유 수(현재)  : {single[multi_mask].nunique():,}개")
print(f"  멀티 고유 수(정렬후): {sorted_preview[multi_mask].nunique():,}개")

현재 고유 컬러 수     : 27,629개
정렬 후 예상 고유 수  : 15,360개  (감소 12,269개)

단품 컬러 행 수       : 365,853건
멀티컬러 행 수        : 186,029건
  멀티 고유 수(현재)  : 26,441개
  멀티 고유 수(정렬후): 14,172개


In [103]:
# 멀티컬러 정렬 통일 적용 (블랙,화이트 / 화이트,블랙 → 블랙,화이트)
df['purchase_option_color'] = df['purchase_option_color'].apply(
    lambda s: ','.join(sorted(s.split(','))) if pd.notna(s) and ',' in s else s
)

print(f"고유 컬러 수: {df['purchase_option_color'].nunique():,}개")
print()
print(df['purchase_option_color'].value_counts().head(20))

고유 컬러 수: 15,360개

purchase_option_color
블랙              110057
블랙,블랙            15552
화이트              14311
블루                4681
네이비               4140
스톤블루              3603
얼스베이지             3556
화이트머스크            3076
블랙,화이트            2787
프레쉬크림             2779
모카베이지             2588
그레이               2465
멜란지그레이            2409
믹스                2330
블랙,앤트러사이트그레이      2305
스톤카키              2291
애쉬블루              2285
오보시디안그레이          2200
실버베이지             2197
블랙,화이트머스크         2152
Name: count, dtype: int64


In [104]:
df['purchase_option_color'].value_counts()

purchase_option_color
블랙                          110057
블랙,블랙                        15552
화이트                          14311
블루                            4681
네이비                           4140
                             ...  
그레이시라벤더,블랙,탠베이지,헤이즈블루            1
그레이시라벤더,코지핑크,톤업베이지,핑크베이지         1
클린그레이,프레쉬크림                      1
프레쉬크림,피치블루                       1
블루,스카이블루                         1
Name: count, Length: 15360, dtype: int64

In [105]:
all_colors = df['purchase_option_color'].dropna().str.split(',').explode()
print(f"총 고유 컬러 수: {all_colors.nunique():,}개")
print()
print(all_colors.value_counts().head())

총 고유 컬러 수: 1,371개

purchase_option_color
블랙        236166
화이트        34393
핑크베이지      12771
화이트머스크      9457
프레쉬크림       9344
Name: count, dtype: int64


In [106]:
# 단일 컬러 기준 이상값 탐지
all_colors = df['purchase_option_color'].dropna().str.split(',').explode().str.strip()
all_colors = all_colors[all_colors != '']

vc = all_colors.value_counts()
print(f"총 고유 컬러 수: {vc.shape[0]:,}개\n")

# ① 숫자 포함
num_mask = all_colors.str.contains(r'\d', regex=True)
print(f"[숫자 포함] {all_colors[num_mask].nunique():,}종  {num_mask.sum():,}건")
print(all_colors[num_mask].value_counts().head(10))

# ② 특수문자 포함 (한글/영문/숫자 외)
sp_mask = all_colors.str.contains(r'[^\w가-힣]', regex=True)
print(f"\n[특수문자 포함] {all_colors[sp_mask].nunique():,}종  {sp_mask.sum():,}건")
print(all_colors[sp_mask].value_counts().head(10))

# ③ 1글자 이하
short_mask = all_colors.str.len() <= 1
print(f"\n[1글자 이하] {all_colors[short_mask].nunique():,}종  {short_mask.sum():,}건")
print(all_colors[short_mask].value_counts())

# ④ 3건 이하 희소 컬러
print(f"\n[3건 이하 희소 컬러] {(vc <= 3).sum():,}개")
print(vc[vc <= 3].head(20))

총 고유 컬러 수: 1,371개

[숫자 포함] 0종  0건
Series([], Name: count, dtype: int64)

[특수문자 포함] 0종  0건
Series([], Name: count, dtype: int64)

[1글자 이하] 0종  0건
Series([], Name: count, dtype: int64)

[3건 이하 희소 컬러] 66개
purchase_option_color
시나몬핑크      3
글램엠파로블루    3
바이올렛도브     3
코튼캔디핑크     3
블러쉬블루      3
프릴옴브레핑크    3
글라시에블루     3
스토미카키      3
트와일라잇민트    3
브론즈베이지     3
클라우디핑크     3
드리프터카키     3
포그네이비      3
크리미피오니     3
글래시얼그레이    3
솔리드커피빈     3
라이트스카이     3
스테그비틀      3
샤인레드       2
마카다미아      2
Name: count, dtype: int64


In [107]:
# # ──────────────────────────────────────────────────
# # purchase_option_size 후처리
# # 실제 사이즈가 아닌 핏/기장 단어를 별도 토큰으로 제거
# # ──────────────────────────────────────────────────
# _drop = re.compile(
#     r'^(숏|롱|미니|스탠다드핏|스탠다드|슬림핏|오버핏|볼륨업|퍼포먼스|후크형|노후크형|맨즈|HOLE|SOLID|9부|8\.2부|4\.5부|3\.5부)$'
# )
# _ws          = re.compile(r'\s+')
# _paren_open  = re.compile(r'\(\s+')
# _paren_close = re.compile(r'\s+\)')

# def clean_size_token(s):
#     s = s.strip()
#     s = _paren_open.sub('(', s)
#     s = _paren_close.sub(')', s)
#     s = _ws.sub(' ', s)
#     return s

# def clean_size(s):
#     if pd.isna(s) or s == '':
#         return pd.NA
#     tokens = [clean_size_token(t) for t in s.split(',')]
#     tokens = [t for t in tokens if t and not _drop.match(t)]
#     return ','.join(tokens) if tokens else pd.NA

# df['purchase_option_size'] = df['purchase_option_size'].apply(clean_size)

# print(f"non-null       : {df['purchase_option_size'].notna().sum():,}건 ({df['purchase_option_size'].notna().mean():.1%})")
# print(f"고유 사이즈 조합 수 : {df['purchase_option_size'].nunique():,}개")
# print()

# # 단일 토큰 기준 분석
# all_sz = df['purchase_option_size'].dropna().str.split(',').explode().str.strip()
# all_sz = all_sz[all_sz != '']
# vc = all_sz.value_counts()
# print(f"총 고유 개별 사이즈 수: {vc.shape[0]:,}개\n")
# print("=== 상위 30개 ===")
# print(vc.head(30))
# print()

# # 이상값 탐지
# sp_mask   = all_sz.str.contains(r'[\[\]=]', regex=True)
# long_mask = all_sz.str.len() > 20
# kor_only  = all_sz.str.fullmatch(r'[가-힣]+', na=False)   # 한글만 (숏/롱 잔재 확인)

# print(f"[특수문자 잔재] {sp_mask.sum():,}건")
# if sp_mask.sum(): print(all_sz[sp_mask].value_counts().head(10))

# print(f"[20자 초과]     {long_mask.sum():,}건")
# if long_mask.sum(): print(all_sz[long_mask].value_counts().head(10))

# print(f"[한글만 (핏/기장 잔재 의심)] {kor_only.sum():,}건")
# if kor_only.sum(): print(all_sz[kor_only].value_counts().head(20))

# print(f"\n[3건 이하 희소 사이즈] {(vc <= 3).sum():,}개")
# print(vc[vc <= 3].head(20))

In [108]:
# ──────────────────────────────────────────────────
# purchase_option_size 후처리
# 실제 사이즈가 아닌 핏/기장/패턴 단어를 개별 토큰으로 인식하여 제거
# ──────────────────────────────────────────────────
_drop = re.compile(
    r'^(숏|롱|미니|스탠다드핏|스탠다드|슬림핏|오버핏|볼륨업|퍼포먼스|후크형|노후크형|맨즈|HOLE|SOLID|9부|8\.2부|4\.5부|3\.5부|솔리드|멜란지|와이드|레귤러|내추럴|엑스트라롱)$'
)
_ws          = re.compile(r'\s+')
_paren_open  = re.compile(r'\(\s+')
_paren_close = re.compile(r'\s+\)')

def clean_size_token(s):
    s = str(s).strip()
    s = _paren_open.sub('(', s)
    s = _paren_close.sub(')', s)
    s = _ws.sub(' ', s)
    return s

def clean_size(s):
    if pd.isna(s) or s == '':
        return pd.NA
    
    # 콤마 단위로 분리하여 토큰별 정제
    tokens = [clean_size_token(t) for t in str(s).split(',')]
    
    # 빈 문자열과 _drop 리스트에 정확히 일치하는 토큰 필터링
    tokens = [t for t in tokens if t and not _drop.match(t)]
    
    return ','.join(tokens) if tokens else pd.NA

# 컬럼에 함수 적용
df['purchase_option_size'] = df['purchase_option_size'].apply(clean_size)

# ─────────────────────────────────────────────
# 사이즈 컬럼 검증 로직 결과 출력
# ─────────────────────────────────────────────
print(f"purchase_option_size non-null : {df['purchase_option_size'].notna().sum():,}건 ({df['purchase_option_size'].notna().mean():.1%})")
print(f"고유 사이즈 조합 수 : {df['purchase_option_size'].nunique():,}개\n")

print("=== 사이즈 상위 30개 ===")
# 단일 토큰 기준으로 쪼개어(explode) 분석
all_sz = df['purchase_option_size'].dropna().str.split(',').explode().str.strip()
all_sz = all_sz[all_sz != '']
vc = all_sz.value_counts()
print(f"총 고유 개별 사이즈 수: {vc.shape[0]:,}개\n")
print(vc.head(30))
print()

# 이상값 탐지 마스킹
sp_mask   = all_sz.str.contains(r'[\[\]=]', regex=True)
long_mask = all_sz.str.len() > 20
kor_only  = all_sz.str.fullmatch(r'[가-힣]+', na=False)   # 한글만 있는 경우 (잔재 여부)

print(f"[특수문자 잔재] {sp_mask.sum():,}건")
if sp_mask.sum() > 0: 
    print(all_sz[sp_mask].value_counts().head(10))

print(f"\n[20자 초과]     {long_mask.sum():,}건")
if long_mask.sum() > 0: 
    print(all_sz[long_mask].value_counts().head(10))

print(f"\n[한글만 (잔재 의심)] {kor_only.sum():,}건")
if kor_only.sum() > 0: 
    print(all_sz[kor_only].value_counts().head(20))

purchase_option_size non-null : 551,882건 (86.4%)
고유 사이즈 조합 수 : 2,448개

=== 사이즈 상위 30개 ===
총 고유 개별 사이즈 수: 223개

purchase_option_size
M (55반~66)             154194
L (66반~77)              83286
S (44~55)               50266
F (Free)                48187
S (55)                  43779
L (100)                 43660
XL (105)                34236
XL (88)                 33135
M (95)                  27247
L (30~32)               20612
XL (32~34)              19531
4 (55반~66)              14787
2XL (110)               14408
2 (44~55)               11939
L (66~77)               11073
XS (44)                 10175
M (70C/75B~C/80A~B)      9799
M                        9352
L                        8566
2XL (34~36)              8332
M (28~30)                7699
S                        7580
6 (66반~77)               6411
L (75C/80B~C/85A)        5470
one size                 5351
M (225~240)              4872
S (70A~B/75A)            4666
L (95)                   4438
S (90)                   436

In [109]:
# 삭제 전 건수 기록
before_count = len(df)

# 결측치를 빈 문자열로 처리한 후, 길이가 5 초과인 데이터만 남기기 (부정 연산자 ~ 대신 직관적인 조건 사용)
df = df[df['content'].fillna('').str.len() > 5]

# 삭제 후 건수 기록
after_count = len(df)

print(f"삭제 전 데이터 건수: {before_count:,}건")
print(f"삭제 후 데이터 건수: {after_count:,}건")
print(f"삭제된 데이터 건수: {before_count - after_count:,}건")

삭제 전 데이터 건수: 638,998건
삭제 후 데이터 건수: 638,989건
삭제된 데이터 건수: 9건


In [114]:
# 1. 대소문자 무시(?i) 및 다양한 프리사이즈 패턴 정의
pattern_free = r'(?i)(F\s*\(Free\)|one\s*size|\bF\b|\bfree\b)'

# 2. 문자열 내 해당 패턴을 모두 '000'으로 일괄 치환 (콤마 유지)
df['purchase_option_size'] = df['purchase_option_size'].str.replace(pattern_free, '000', regex=True)

# 3. 검증
print(f"고유 사이즈 조합 수 : {df['purchase_option_size'].nunique():,}개\n")
print(df['purchase_option_size'].value_counts().head(30))

고유 사이즈 조합 수 : 2,446개

purchase_option_size
M (55반~66)                          65892
000                                 41801
M (55반~66),M (55반~66)               29434
L (66반~77)                          28848
S (44~55)                           27527
L (66반~77),L (66반~77)               17980
S (55)                              17586
L (30~32)                           13556
4 (55반~66)                          12876
XL (32~34)                          12433
XL (88)                             11714
2 (44~55)                           11186
L (66~77)                            9129
L (100),L (100),L (100)              8465
S (44~55),S (44~55)                  8277
L (100)                              7538
XL (88),XL (88)                      7398
XL (105),XL (105),XL (105)           7010
S (55),S (55)                        6855
XL (105)                             6018
000,000                              5819
M (55반~66),M (55반~66),M (55반~66)     5609
6 (66반~77)                       

In [116]:
# 여성사이즈 XS(80), S(85), M(90), L(95), XL(100), 2XL(105), 3XL(110) 등 확인 및 WXS(80), WS(85), WM(90), WL(95), WXL(100), W2XL(105), W3XL(110) 등으로 치환

# 1. 여성 사이즈 판별 정규식 (공용 사이즈 보호)
# 알파벳 뒤에 (80), (85) 등 특정 숫자가 붙어있는 경우만 매칭 그룹(\1)으로 캡처
# W가 이미 붙어있다면(W?) 무시하고 알파벳부터 캡처합니다.
pattern_women = r'(?i)\bW?(XS\s*\(\s*80\s*\)|S\s*\(\s*85\s*\)|M\s*\(\s*90\s*\)|L\s*\(\s*95\s*\)|XL\s*\(\s*100\s*\)|2XL\s*\(\s*105\s*\)|3XL\s*\(\s*110\s*\))'

# 2. 파생 컬럼 생성: women_size_yn (1=True / 0=False)
df['women_size_yn'] = df['purchase_option_size'].str.contains(pattern_women, na=False).astype(int)

# 3. 사이즈 치환: 캡처된 그룹(\1) 앞에 강제로 'W'를 붙임
# 예: 'S (85)' -> 'WS (85)', 'WS (85)' -> 'WS (85)' / 단독 'S'나 'M (95)'는 무시됨
df['purchase_option_size'] = (
    df['purchase_option_size']
    .str.replace(pattern_women, r'W\1', regex=True)
)

# ─────────────────────────────────────────────
# 결과 확인
# ─────────────────────────────────────────────
print(f"여성 사이즈(women_size_yn=1) 건수: {df['women_size_yn'].sum():,}건")
print("\n[치환된 사이즈 샘플]")
print(df.loc[df['women_size_yn'] == 1, 'purchase_option_size'].value_counts().head(10))

# ─────────────────────────────────────────────
# 공용/기타 사이즈 보호 확인
# ─────────────────────────────────────────────
print("\n[보호된 공용/기타 사이즈 샘플 (W가 안 붙어야 정상)]")
mask_unisex = (df['women_size_yn'] == 0) & df['purchase_option_size'].str.contains(r'\b(S|M|L)\b', na=False)
print(df.loc[mask_unisex, 'purchase_option_size'].value_counts().head(10))


여성 사이즈(women_size_yn=1) 건수: 3,371건

[치환된 사이즈 샘플]
purchase_option_size
WL (95),WL (95),WL (95),WL (95),M (70C/75B~C/80A~B),M (70C/75B~C/80A~B),M (70C/75B~C/80A~B),M (70C/75B~C/80A~B)    131
WM (90),WM (90),WM (90),WM (90),M (70C/75B~C/80A~B),M (70C/75B~C/80A~B),M (70C/75B~C/80A~B),M (70C/75B~C/80A~B)    101
WL (95),WL (95),WL (95),WL (95),L (75C/80B~C/85A),L (75C/80B~C/85A),L (75C/80B~C/85A),L (75C/80B~C/85A)             97
WL (95),WL (95),WL (95),WL (95),WL (95),WL (95)                                                                     90
WL (95),WL (95),WL (95)                                                                                             79
WM (90),WM (90),WM (90),WM (90),WM (90),WM (90)                                                                     69
WM (90),WM (90),WM (90)                                                                                             61
M (70C/75B~C/80A~B),M (70C/75B~C/80A~B),WM (90),WM (90)                                          

In [117]:
# ========================
# 최종 컬럼 정리
# ========================

# 1) 데이터 형변환
def convert_dtypes(df):
    # 날짜 컬럼
    for col in ["collect_date", "review_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # ID 컬럼
    for col in ["review_id", "product_id"]:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # 문자열 컬럼
    for col in ["product_name", "content", "purchase_option", "product_url"]:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # category 컬럼
    for col in ["platform", "purchase_option_color", "purchase_option_size", "user_height_group", "user_weight_group"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # 정수형 컬럼
    for col, dtype in [("year", "Int16"), ("month", "Int8"), ("rating", "Int8"), ("helpful_count", "Int32")]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(dtype)

    # 0/1 플래그 컬럼
    for col in ["has_image", "women_size_yn"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int8")

    # 실수형 컬럼
    for col in ["user_height", "user_weight"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    return df

# 사용
df = convert_dtypes(df)
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
Index: 638989 entries, 0 to 638997
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   collect_date           638989 non-null  datetime64[us]
 1   platform               638989 non-null  category      
 2   review_id              638989 non-null  string        
 3   product_id             638989 non-null  string        
 4   product_name           638989 non-null  string        
 5   review_date            638989 non-null  datetime64[us]
 6   year                   638989 non-null  Int16         
 7   month                  638989 non-null  Int8          
 8   content                638989 non-null  string        
 9   rating                 638989 non-null  Int8          
 10  helpful_count          638989 non-null  Int32         
 11  has_image              638989 non-null  int8          
 12  purchase_option        551873 non-null  string        
 13  

In [118]:
# ======================
# 컬럼 순서 재정렬
# ======================

column_order = [
    'collect_date',
    'platform',
    'review_id',
    'product_id',
    'product_name',
    'review_date',
    'year',
    'month',
    'content',
    'rating',
    'helpful_count',
    'has_image',
    'purchase_option',
    'purchase_option_color',
    'purchase_option_size',
    'women_size_yn',
    'user_height',
    'user_weight',
    'user_height_group',
    'user_weight_group',
    'product_url'
]

# 컬럼 존재하는 것만 선택 (안전하게)
df = df[
    [col for col in column_order if col in df.columns]
]

# 확인
df.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-21,자사몰,48724074,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-14,2026,4,"러닝 후 몸 풀어주는 용으로 아주 좋아요. 종아리나 등, 목 뒤에 사용하기 좋아요....",5,0,1,"[EJFMB-02DPK] 샤이 핑크, F (Free)",샤이핑크,000,0,NaN,NaN,155~159cm,45~49kg,https://andar.co.kr/product/detail.html?produc...
1,2026-04-21,자사몰,48590531,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-07,2026,4,딱딱하네요\n저는 말랑말랑 한줄 알았어요(2026-04-07 20:15:59 에 등...,3,0,0,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN,https://andar.co.kr/product/detail.html?produc...
2,2026-04-21,자사몰,48535840,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-05,2026,4,요가링이랑 폼롤러랑 같이 사서 집에서 잘 굴려주고 있어요 색깔도 너무 예쁘고 귀여워...,5,0,1,"[EJFMB-02DPK] 샤이 핑크, F (Free)",샤이핑크,000,0,NaN,NaN,165~169cm,55~59kg,https://andar.co.kr/product/detail.html?produc...
3,2026-04-21,자사몰,48535469,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-05,2026,4,묵직하니 사용하기 편하네요 릴렉스 마사지 듀얼볼 Ⅱ 후기 정사이즈,5,0,0,"[EJFMB-02DPK] 샤이 핑크, F (Free)",샤이핑크,000,0,NaN,NaN,160~164cm,50~54kg,https://andar.co.kr/product/detail.html?produc...
4,2026-04-21,자사몰,48521693,5147,릴렉스 마사지 듀얼볼 Ⅱ,2026-04-03,2026,4,시원하게 마사지가되요 여기저기 쉽게 문지르고 아픈 부위 꾹꾹 눌러주니 시원하네요 릴...,5,0,1,"[EJFMB-02DPK] 샤이 핑크, F (Free)",샤이핑크,000,0,NaN,NaN,160~164cm,55~59kg,https://andar.co.kr/product/detail.html?produc...


In [ ]:
# df.to_csv('./final_data/andar_homepage_review_final_s2024.csv', index=False)